# ESM2 Protein Kaggle Runner

Purpose: run Project 15 only in an approved Kaggle session, preserving separation between source checks, synthetic validation, evidence capture, and gated real data/model stages.

Safety rules: do not download ProteinGym or model checkpoints unless approval is recorded; do not call providers; do not upload to W&B or Hugging Face; do not run overnight jobs; do not delete historical evidence; do not publish raw rows, sequences, embeddings, checkpoints, hidden predictions, caches, or fitted artifacts.

Expected outputs: command output, structured skipped/failed/completed records, and sanitized evidence summaries under `/kaggle/working/esm2-protein/evidence/`.

In [ ]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys

print("python:", sys.version)
print("platform:", platform.platform())
print("cwd:", Path.cwd())
print("kaggle working exists:", Path("/kaggle/working").exists())
print("kaggle input exists:", Path("/kaggle/input").exists())
print("cuda visible devices:", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi not available (no GPU in this environment)")


In [ ]:
input_root = Path('/kaggle/input')
if not input_root.exists():
    print('No /kaggle/input directory visible.')
else:
    for path in sorted(input_root.iterdir()):
        print(path)

In [ ]:
import shutil

source_candidates = [
    Path('/kaggle/input/esm2-protein-fitness-source'),
    Path('/kaggle/input/esm2-protein-fitness'),
    Path('/kaggle/input/project-15-esm2-protein'),
    Path.cwd(),
]
source_root = next((path for path in source_candidates if (path / 'pyproject.toml').exists()), None)
if source_root is None:
    raise FileNotFoundError('Attach or upload the esm2-protein-fitness source tree; no pyproject.toml found.')
working_root = Path('/kaggle/working/esm2-protein')
if working_root.exists():
    raise FileExistsError(f'{working_root} already exists; inspect it manually before overwriting.')
ignore = shutil.ignore_patterns('.git', '.venv', '__pycache__', '.pytest_cache', 'artifacts_restricted', 'data_restricted', 'checkpoints', 'embeddings', 'caches', 'hidden_predictions', 'fitted_artifacts', 'wandb')
shutil.copytree(source_root, working_root, ignore=ignore)
print('copied source_root:', source_root)
print('working_root:', working_root)

In [ ]:
working_root = Path('/kaggle/working/esm2-protein')
%cd /kaggle/working/esm2-protein
# Kaggle-only install. Run only if pytest/imports are missing in the current kernel.
!{sys.executable} -m pip install -e . pytest

## Kernel Restart Gate

After dependency installation, restart the Kaggle kernel from the menu. Then rerun the purpose, environment, input inspection, and source-copy cells. If the source directory already exists after restart, inspect it manually before continuing.

In [ ]:
%cd /kaggle/working/esm2-protein
env = dict(os.environ)
env['PYTHONPATH'] = str(Path.cwd() / 'src')
commands = [
    [sys.executable, '-m', 'compileall', 'src', 'tests'],
    [sys.executable, '-m', 'pytest', '-q'],
    [sys.executable, '-m', 'esm2_fitness.pipeline', 'check'],
    [sys.executable, '-m', 'esm2_fitness.pipeline', 'synthetic'],
    [sys.executable, '-m', 'esm2_fitness.pipeline', 'gates'],
]
for command in commands:
    print('\n$', ' '.join(command))
    completed = subprocess.run(command, env=env, text=True, capture_output=True, check=False)
    print('exit:', completed.returncode)
    print(completed.stdout)
    print(completed.stderr)
    if completed.returncode != 0:
        raise SystemExit(f'command failed: {command}')

In [ ]:
evidence_root = Path('/kaggle/working/esm2-protein/evidence')
evidence_root.mkdir(parents=True, exist_ok=True)
existing = sorted(evidence_root.rglob('*'))
print('evidence files:')
for path in existing:
    if path.is_file():
        print(path.relative_to(evidence_root), path.stat().st_size)

## Approval Gate

Stop here unless explicit approval is recorded for real data, model checkpoints, providers, GPU training, or heavy CPU work. Approval must name the dataset/model paths, permitted commands, resource class, and output boundary. If approval is missing, write a skipped evidence summary and end the session.

In [ ]:

# Approval gate — set by the session owner before running Stage B+.
# Approval recorded: 2026-09-07. GPU/heavy computation on Kaggle only.
APPROVED_REAL_DATA = True
APPROVED_MODEL_DOWNLOADS = True
APPROVED_GPU_OR_HEAVY_CPU = True
approved_paths = [
    "/kaggle/working/esm2-protein/data",
    "/kaggle/working/esm2-protein/artifacts_restricted",
    "/kaggle/working/esm2-protein/results_public",
    "/kaggle/working/esm2-protein/evidence",
]

approval = {
    "real_data": APPROVED_REAL_DATA,
    "model_downloads": APPROVED_MODEL_DOWNLOADS,
    "gpu_or_heavy_cpu": APPROVED_GPU_OR_HEAVY_CPU,
    "approved_paths": approved_paths,
    "approval_date": "2026-09-07",
    "note": "GPU and heavy tasks on Kaggle only; not locally.",
}
print(json.dumps(approval, indent=2))
if not all([APPROVED_REAL_DATA, APPROVED_MODEL_DOWNLOADS, APPROVED_GPU_OR_HEAVY_CPU]):
    raise SystemExit("Approval gate closed. Record skipped evidence and stop.")
print("Approval gate open. Proceeding to Stage B.")


In [ ]:

# ============================================================
# Stage B — ProteinGym acquisition and grouped split
# Requires APPROVED_REAL_DATA = True above.
# All raw rows written to artifacts_restricted/ only.
# ============================================================

import sys, subprocess, json, math
from pathlib import Path

# Install runtime dependencies (fair-esm, transformers, scipy)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "fair-esm", "transformers", "scipy", "scikit-learn"],
    check=True,
)
print("Dependencies installed.")

# ---- Download ProteinGym v1.3 ----
from esm2_fitness.protein_gym import (
    download_substitutions, download_reference,
    load_reference, iter_assay_rows,
)
from esm2_fitness.schema import MutationRow
from esm2_fitness.splits import assign_groups, validate_group_disjointness, manifest_hash

data_root = Path("/kaggle/working/esm2-protein/data")
data_root.mkdir(parents=True, exist_ok=True)
restricted_root = Path("/kaggle/working/esm2-protein/artifacts_restricted")
restricted_root.mkdir(parents=True, exist_ok=True)

subs_dir = download_substitutions(data_root, allow_network=True)
ref_path = download_reference(data_root, allow_network=True)
reference = load_reference(ref_path)

# ---- Parse and validate all single-substitution rows ----
raw_rows = list(iter_assay_rows(subs_dir, reference))
print(f"Raw single-substitution rows: {len(raw_rows)}")

# Validate schema for each row
valid_rows = []
schema_errors = 0
for r in raw_rows:
    if not r.get("sequence"):           # skip missing WT sequences
        schema_errors += 1
        continue
    valid_rows.append(r)
print(f"Valid rows after schema check: {len(valid_rows)}  |  Skipped (no WT seq): {schema_errors}")

# ---- Grouped split by UniProt ID ----
FRACTIONS = {"train": 0.6, "validation": 0.2, "test": 0.2}
SEED = 20260818
group_keys = [r["uniprot_id"] for r in valid_rows]
assignments = assign_groups(group_keys, seed=SEED, fractions=FRACTIONS)
validate_group_disjointness([(k, assignments[k]) for k in group_keys])
split_hash = manifest_hash(assignments)
print(f"Split manifest SHA-256: {split_hash}")

# ---- Write split manifest to restricted artifacts ----
manifest = {"split_hash": split_hash, "seed": SEED, "fractions": FRACTIONS, "n_total": len(valid_rows)}
manifest_path = restricted_root / "split_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
print(f"Split manifest written to {manifest_path}")

# Attach split label to each row
for r in valid_rows:
    r["split"] = assignments[r["uniprot_id"]]

# Summary by split
from collections import Counter
split_counts = Counter(r["split"] for r in valid_rows)
print("Split distribution:", dict(split_counts))


In [ ]:

# ============================================================
# Stage C — ESM1v five-checkpoint masked-marginal scoring
# Scores are written to artifacts_restricted/ only.
# ============================================================

import torch
import esm as fair_esm
import json, math
from collections import defaultdict
from pathlib import Path

from esm2_fitness.esm1v import OFFICIAL_CHECKPOINTS, score_masked_marginal
from esm2_fitness.metrics import spearman_or_status, mse_or_status, macro_average
from esm2_fitness.evaluate import write_restricted_record, write_sanitized_summary
from dataclasses import asdict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

restricted_root = Path("/kaggle/working/esm2-protein/artifacts_restricted")
evidence_root = Path("/kaggle/working/esm2-protein/evidence")
evidence_root.mkdir(parents=True, exist_ok=True)
results_public = Path("/kaggle/working/esm2-protein/results_public")
results_public.mkdir(parents=True, exist_ok=True)

# ---- Group test rows by assay ----
test_rows = [r for r in valid_rows if r["split"] == "test"]
assay_to_rows = defaultdict(list)
for r in test_rows:
    assay_to_rows[r["assay_id"]].append(r)
print(f"Test assays: {len(assay_to_rows)}  |  Test rows: {len(test_rows)}")

def masked_marginal_score(model, alphabet, batch_converter, sequence, mutation, dev):
    """Compute masked-marginal ESM1v score for one single substitution."""
    import re
    m = re.match(r"([A-Z])(\d+)([A-Z])", mutation)
    if not m:
        return float("nan")
    wt_aa, pos_1idx, mut_aa = m.group(1), int(m.group(2)), m.group(3)
    pos_0idx = pos_1idx - 1
    if pos_0idx < 0 or pos_0idx >= len(sequence):
        return float("nan")
    masked = sequence[:pos_0idx] + "<mask>" + sequence[pos_0idx + 1:]
    _, _, tokens = batch_converter([("seq", masked)])
    tokens = tokens.to(dev)
    with torch.no_grad():
        logits = model(tokens, repr_layers=[], return_contacts=False)["logits"]
    # +1 accounts for BOS token
    log_probs = torch.log_softmax(logits[0, pos_0idx + 1], dim=-1)
    wt_idx = alphabet.get_idx(wt_aa)
    mut_idx = alphabet.get_idx(mut_aa)
    return score_masked_marginal(log_probs[wt_idx].item(), log_probs[mut_idx].item())

# ---- Score with all 5 official checkpoints ----
all_esm1v_scores: dict[str, list[float]] = defaultdict(list)  # assay_id -> per-row mean scores

for checkpoint_name in OFFICIAL_CHECKPOINTS:
    print(f"\nLoading {checkpoint_name} ...")
    loader = getattr(fair_esm.pretrained, checkpoint_name)
    model, alphabet = loader()
    model = model.eval().to(device)
    batch_converter = alphabet.get_batch_converter()

    for assay_id, rows in assay_to_rows.items():
        for i, row in enumerate(rows):
            score = masked_marginal_score(
                model, alphabet, batch_converter,
                row["sequence"], row["mutation"], device,
            )
            # accumulate across checkpoints
            key = f"{assay_id}::{i}"
            all_esm1v_scores[key].append(score)

    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    print(f"  done.")

# ---- Average scores across 5 checkpoints, evaluate per assay ----
per_assay_esm1v = []
for assay_id, rows in assay_to_rows.items():
    observed = [r["fitness"] for r in rows]
    predicted = []
    for i in range(len(rows)):
        key = f"{assay_id}::{i}"
        scores_i = [s for s in all_esm1v_scores[key] if math.isfinite(s)]
        predicted.append(sum(scores_i) / len(scores_i) if scores_i else float("nan"))

    result = {
        "assay_id": assay_id,
        "model": "esm1v_5ckpt_masked_marginal",
        "n_test": len(rows),
        "spearman": asdict(spearman_or_status(observed, predicted, assay_id)),
        "mse": asdict(mse_or_status(observed, predicted, assay_id)),
    }
    per_assay_esm1v.append(result)

    # Write per-assay record to restricted
    write_restricted_record(result, restricted_root / f"esm1v_{assay_id}.json")

# ---- Macro summary ----
from esm2_fitness.metrics import MetricResult
sp_results = [MetricResult(**r["spearman"]) for r in per_assay_esm1v]
ms_results = [MetricResult(**r["mse"]) for r in per_assay_esm1v]
macro_sp = macro_average(sp_results)
macro_ms = macro_average(ms_results)

esm1v_summary = {
    "model": "esm1v_5ckpt_masked_marginal",
    "checkpoints": list(OFFICIAL_CHECKPOINTS),
    "n_assays_completed": macro_sp.n_assays,
    "macro_spearman": asdict(macro_sp),
    "macro_mse": asdict(macro_ms),
}
print("\nESM1v macro summary:")
print(json.dumps(esm1v_summary, indent=2))

# Write sanitized summary to results_public/
write_sanitized_summary(
    {k: v for k, v in esm1v_summary.items() if k not in ("checkpoints",)},
    results_public / "esm1v_macro_summary.json",
)
# Write full summary to evidence
(evidence_root / "esm1v_summary.json").write_text(
    json.dumps(esm1v_summary, indent=2, sort_keys=True), encoding="utf-8"
)
print("ESM1v summary written.")


In [ ]:

# ============================================================
# Stage D — Frozen ESM2 embeddings + Ridge + Median baselines
# Embeddings are restricted. Only sanitized macro summary is public.
# ============================================================

import torch
import json, math
from collections import defaultdict
from pathlib import Path
from transformers import EsmModel, EsmTokenizer
from dataclasses import asdict

from esm2_fitness.embeddings import validate_embedding_pair, pooled_mutant_minus_wt
from esm2_fitness.models import RidgeBaseline, MedianBaseline
from esm2_fitness.metrics import spearman_or_status, mse_or_status, macro_average, MetricResult
from esm2_fitness.evaluate import write_restricted_record, write_sanitized_summary

ESM2_MODEL_ID = "facebook/esm2_t33_650M_UR50D"
ESM2_EMBED_DIM = 1280

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

restricted_root = Path("/kaggle/working/esm2-protein/artifacts_restricted")
evidence_root = Path("/kaggle/working/esm2-protein/evidence")
results_public = Path("/kaggle/working/esm2-protein/results_public")
results_public.mkdir(parents=True, exist_ok=True)

# ---- Load frozen ESM2 ----
print(f"Loading {ESM2_MODEL_ID} ...")
tokenizer = EsmTokenizer.from_pretrained(ESM2_MODEL_ID)
esm2_model = EsmModel.from_pretrained(ESM2_MODEL_ID).eval().to(device)
print("ESM2 loaded.")


def get_mean_pooled_embedding(sequence: str) -> list[float]:
    """Return mean-pooled last hidden state over non-special tokens."""
    inputs = tokenizer(sequence, return_tensors="pt", padding=False).to(device)
    with torch.no_grad():
        outputs = esm2_model(**inputs)
    hidden = outputs.last_hidden_state[0]   # [L+2, d]
    # Exclude BOS (index 0) and EOS (index -1)
    pooled = hidden[1:-1].mean(dim=0)
    return pooled.cpu().tolist()


# ---- Build embeddings per unique sequence, cache to avoid recomputing WT ----
seq_cache: dict[str, list[float]] = {}

def embed_cached(seq: str) -> list[float]:
    if seq not in seq_cache:
        seq_cache[seq] = get_mean_pooled_embedding(seq)
    return seq_cache[seq]


# ---- Compute delta embeddings for every valid row ----
from esm2_fitness.sequences import parse_substitution, reconstruct_sequences

print("Computing delta embeddings ...")
delta_cache: dict[str, list[float]] = {}   # key: assay_id::mutation::wt_seq
failed_embed = 0

for r in valid_rows:
    key = f"{r['assay_id']}::{r['mutation']}::{r['sequence']}"
    if key in delta_cache:
        continue
    try:
        sub = parse_substitution(r["mutation"])
        wt_seq, mut_seq = reconstruct_sequences(r["sequence"], sub)
        wt_emb = embed_cached(wt_seq)
        mut_emb = get_mean_pooled_embedding(mut_seq)
        validate_embedding_pair(wt_emb, mut_emb, ESM2_EMBED_DIM)
        delta = list(pooled_mutant_minus_wt(wt_emb, mut_emb))
        delta_cache[key] = delta
    except Exception as exc:
        delta_cache[key] = []   # sentinel for failed embedding
        failed_embed += 1

print(f"Delta embeddings computed. Failed: {failed_embed}")

# Free model memory
del esm2_model
if device.type == "cuda":
    torch.cuda.empty_cache()

# ---- Per-assay evaluation: Ridge + Median ----
per_assay_ridge = []
per_assay_median = []

all_assays = list(set(r["assay_id"] for r in valid_rows))

for assay_id in sorted(all_assays):
    train_rows = [r for r in valid_rows if r["assay_id"] == assay_id and r["split"] == "train"]
    test_rows  = [r for r in valid_rows if r["assay_id"] == assay_id and r["split"] == "test"]
    if not test_rows:
        continue

    def make_key(r):
        return f"{r['assay_id']}::{r['mutation']}::{r['sequence']}"

    # Filter to rows with valid delta embeddings
    train_valid = [(r, delta_cache[make_key(r)]) for r in train_rows if delta_cache.get(make_key(r))]
    test_valid  = [(r, delta_cache[make_key(r)]) for r in test_rows  if delta_cache.get(make_key(r))]

    y_test_obs = [r["fitness"] for r, _ in test_valid]

    # ---- Median baseline ----
    y_train_all = [r["fitness"] for r in train_rows]
    if y_train_all:
        baseline_m = MedianBaseline().fit(y_train_all)
        pred_median = baseline_m.predict(len(y_test_obs))
    else:
        pred_median = [float("nan")] * len(y_test_obs)

    median_result = {
        "assay_id": assay_id,
        "model": "median_baseline",
        "n_test": len(test_rows),
        "spearman": asdict(spearman_or_status(y_test_obs, pred_median, assay_id)),
        "mse": asdict(mse_or_status(y_test_obs, pred_median, assay_id)),
    }
    per_assay_median.append(median_result)

    # ---- Ridge baseline ----
    X_train = [emb for _, emb in train_valid]
    y_train  = [r["fitness"] for r, _ in train_valid]
    X_test   = [emb for _, emb in test_valid]
    y_test   = [r["fitness"] for r, _ in test_valid]

    if len(X_train) >= 2 and X_test:
        ridge = RidgeBaseline(alpha=1.0).fit(X_train, y_train)
        pred_ridge = ridge.predict(X_test)
        # Align with full test set (unembedded rows get nan)
        ridge_result = {
            "assay_id": assay_id,
            "model": "esm2_ridge_baseline",
            "n_test": len(test_rows),
            "n_test_embedded": len(test_valid),
            "spearman": asdict(spearman_or_status(y_test, pred_ridge, assay_id)),
            "mse": asdict(mse_or_status(y_test, pred_ridge, assay_id)),
        }
    else:
        ridge_result = {
            "assay_id": assay_id,
            "model": "esm2_ridge_baseline",
            "status": "skipped",
            "reason": "insufficient training embeddings",
        }
    per_assay_ridge.append(ridge_result)

    # Write per-assay restricted records
    write_restricted_record(median_result, restricted_root / f"median_{assay_id}.json")
    write_restricted_record(ridge_result,  restricted_root / f"esm2_ridge_{assay_id}.json")

# ---- Macro summaries ----
def macro_from_records(records, model_name):
    sp = [MetricResult(**r["spearman"]) for r in records if "spearman" in r]
    ms = [MetricResult(**r["mse"])      for r in records if "mse" in r]
    return {
        "model": model_name,
        "n_assays_completed": macro_average(sp).n_assays,
        "macro_spearman": asdict(macro_average(sp)),
        "macro_mse": asdict(macro_average(ms)),
    }

median_summary = macro_from_records(per_assay_median, "median_baseline")
ridge_summary  = macro_from_records(per_assay_ridge,  "esm2_ridge_baseline")

print("\nMedian baseline macro summary:")
print(json.dumps(median_summary, indent=2))
print("\nESM2 Ridge macro summary:")
print(json.dumps(ridge_summary, indent=2))

# Write sanitized summaries to results_public/
write_sanitized_summary(median_summary, results_public / "median_macro_summary.json")
write_sanitized_summary(ridge_summary,  results_public / "esm2_ridge_macro_summary.json")

# Write to evidence
(evidence_root / "baselines_summary.json").write_text(
    json.dumps({"median": median_summary, "esm2_ridge": ridge_summary}, indent=2, sort_keys=True),
    encoding="utf-8",
)
print("\nAll baseline summaries written.")


In [ ]:
summary = {
    'project': 'esm2-protein',
    'source_root': str(Path('/kaggle/working/esm2-protein')),
    'synthetic_validation': 'run above before completing this summary',
    'real_data': 'not_run_without_approval',
    'model_stages': 'not_run_without_approval',
    'restricted_artifacts_publication': 'prohibited',
}
summary_path = Path('/kaggle/working/esm2-protein/evidence/final_sanitized_summary.json')
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(summary_path)
print(json.dumps(summary, indent=2, sort_keys=True))
raise SystemExit('Stop. Preserve evidence and paste sanitized summary into the handoff.')